# Zomato Data Analysis Using Python (Synthetic Dataset)

This notebook follows a complete beginner-friendly workflow:
1. Generate a synthetic Zomato-style dataset
2. Clean and validate the data
3. Analyze restaurant trends and customer preferences
4. Visualize insights with charts
5. Answer the three business questions

## Step 1: Import Required Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from create_zomato_dataset import generate_dataset

## Step 2: Create Synthetic Dataset
Generate 500 rows so analysis can run even without an external file.

In [ ]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "zomato_synthetic_dataset.csv"

generate_dataset(output_path=DATA_PATH, row_count=500, seed=42, preview_rows=5, print_preview=True)

## Step 3: Load Dataset and Preview

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Step 4: Data Cleaning and Preparation
- Convert rate from text format (like 4.1/5) to float
- Ensure cost is numeric
- Normalize category values

In [ ]:
def handle_rate(value):
    value = str(value).strip()
    if "/" in value:
        value = value.split("/")[0]
    return float(value)

df["rate"] = df["rate"].apply(handle_rate)
df["approx_cost(for two people)"] = df["approx_cost(for two people)"].astype(str).str.replace(",", "", regex=False).astype(int)
df["listed_in(type)"] = df["listed_in(type)"].astype(str).str.strip()
df["online_order"] = df["online_order"].astype(str).str.strip().str.title()

print(df.info())
print(df.isnull().sum())

## Step 5: Restaurant Type Distribution
This shows which restaurant categories are most common.

In [ ]:
plt.figure(figsize=(10, 5))
order = df["listed_in(type)"].value_counts().index
sns.countplot(x=df["listed_in(type)"], order=order, color="#66c2a5")
plt.xticks(rotation=25)
plt.title("Restaurant Type Distribution")
plt.xlabel("Type of restaurant")
plt.ylabel("Count")
plt.show()

df["listed_in(type)"].value_counts()

## Step 6: Votes by Restaurant Type
Total votes help estimate public preference intensity.

In [ ]:
grouped_votes = df.groupby("listed_in(type)", as_index=False)["votes"].sum().sort_values("votes", ascending=False)

plt.figure(figsize=(10, 5))
sns.lineplot(data=grouped_votes, x="listed_in(type)", y="votes", marker="o", color="green")
plt.xticks(rotation=25)
plt.title("Total Votes by Restaurant Type")
plt.xlabel("Type of restaurant")
plt.ylabel("Total votes")
plt.show()

grouped_votes

## Step 7: Most Voted Restaurant

In [ ]:
max_votes = df["votes"].max()
df.loc[df["votes"] == max_votes, ["name", "listed_in(type)", "votes"]]

## Step 8: Online Order Availability
Compare restaurants accepting online orders vs offline-only.

In [ ]:
plt.figure(figsize=(6, 5))
sns.countplot(data=df, x="online_order", color="#8da0cb")
plt.title("Online vs Offline Order Availability")
plt.xlabel("Online order available")
plt.ylabel("Count")
plt.show()

df["online_order"].value_counts()

## Step 9: Ratings Distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["rate"], bins=8, color="#377eb8", edgecolor="black")
plt.title("Ratings Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of restaurants")
plt.show()

df["rate"].describe()

## Step 10: Couple Budget Preference

In [ ]:
plt.figure(figsize=(11, 5))
cost_order = df["approx_cost(for two people)"].value_counts().sort_index().index
sns.countplot(data=df, x="approx_cost(for two people)", order=cost_order, color="#fdae61")
plt.xticks(rotation=45)
plt.title("Preferred Couple Budget Range")
plt.xlabel("Approx cost for two people")
plt.ylabel("Count")
plt.show()

df["approx_cost(for two people)"].mode().iloc[0]

## Step 11: Ratings by Online vs Offline

In [ ]:
plt.figure(figsize=(7, 6))
sns.boxplot(data=df, x="online_order", y="rate", color="#fc8d62")
plt.title("Ratings by Online Order Availability")
plt.xlabel("Online order")
plt.ylabel("Rating")
plt.show()

df.groupby("online_order")["rate"].median()

## Step 12: Heatmap of Restaurant Type vs Order Mode

In [ ]:
pivot_table = df.pivot_table(index="listed_in(type)", columns="online_order", aggfunc="size", fill_value=0)

plt.figure(figsize=(8, 6))
sns.heatmap(pivot_table, annot=True, cmap="YlGnBu", fmt="d")
plt.title("Type vs Online Order Availability")
plt.xlabel("Online Order")
plt.ylabel("Restaurant Type")
plt.show()

pivot_table

## Final Answers
1. Do more restaurants provide online delivery?
2. Which restaurant types are most favored?
3. What price range do couples prefer?

In [ ]:
online_counts = df["online_order"].value_counts()
most_favored_type = grouped_votes.iloc[0]["listed_in(type)"]
preferred_cost = int(df["approx_cost(for two people)"].mode().iloc[0])

print(f"Online restaurants: {int(online_counts.get('Yes', 0))}")
print(f"Offline-only restaurants: {int(online_counts.get('No', 0))}")
print(f"Most favored type (by total votes): {most_favored_type}")
print(f"Preferred couple budget (mode): {preferred_cost}")